# v73 — GONG SSL → refiner encoder init (fold2 screen)

**Provenance gate:** corpus rebuilt under `train-test-temporal-v2`
with `corpus_config.json`; split CSV + YOLO/baseline checkpoint bytes
must be bound by an independently reviewed manifest (`v73_reviewed_provenance.json`,
set the externally reviewed SHA256 as `V73_PROVENANCE_SHA256`).
残る未試行レーン: 無ラベル GONG 全日面コーパスで **セグメンテーション側**
エンコーダを SimCLR 事前学習し、prod EdgeAtt refiner の fold2 OOF PQ が
scratch init を上回るかを検証する。

**事前固定ゲート** (RESEARCH_LOG §2.26):
- delta(PQ) = ssl − scratch >= +0.010 → 確認実験 (fold0/1 + seed) へ
- 0 <= delta < +0.010 → 棄却 (ノイズ帯)
- delta < 0 → 棄却
- sanity: baseline ckpt の fold2 PQ が ~0.448 から外れすぎないこと


In [ ]:
import os, shutil, subprocess, sys, json
from pathlib import Path
import re
EXPECTED_PROVENANCE_SHA = os.environ.get('V73_PROVENANCE_SHA256', '')
if not EXPECTED_PROVENANCE_SHA:
    try:
        from google.colab import userdata
        EXPECTED_PROVENANCE_SHA = userdata.get('V73_PROVENANCE_SHA256') or ''
    except Exception:
        pass
if not EXPECTED_PROVENANCE_SHA:
    try:
        from kaggle_secrets import UserSecretsClient
        EXPECTED_PROVENANCE_SHA = UserSecretsClient().get_secret('V73_PROVENANCE_SHA256')
    except Exception:
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    try:
        EXPECTED_PROVENANCE_SHA = input('V73_PROVENANCE_SHA256 (reviewed manifest hash): ').strip()
    except (EOFError, OSError):
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    raise RuntimeError('Set V73_PROVENANCE_SHA256 after independent provenance review')

IN = Path('/kaggle/input')
print(sorted(str(p.relative_to(IN)) for p in IN.rglob('*') if p.is_dir())[:40])
corpus_dir = next(p.parent for p in IN.rglob('batch_*.npz'))
preproc_dir = next(p.parent for p in IN.rglob('preprocessed_v30_q96/*.jpeg'))
folds_csv = next(IN.rglob('grouped_folds_train_split.csv'))
import hashlib
EXPECTED_FOLDS_SHA = 'febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e'
actual = hashlib.sha256(folds_csv.read_bytes()).hexdigest()
if actual != EXPECTED_FOLDS_SHA:
    raise RuntimeError(f'fold CSV sha mismatch: {actual}')
print('folds_csv sha256 OK (true experimental split)')
baseline_ckpt = next(IN.rglob('mediseg_edgeattnet_refiner_fold2_resume.pt'))
yolo_ckpt = next(IN.rglob('yolo_fold2_best.pt'))
train_json = next(IN.rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json'))
code_dir = next(p.parent for p in IN.rglob('train_refiner_fold.py'))
sys.path.insert(0, str(code_dir))
# Provenance is a reviewed input, never auto-generated from ambiguous legacy weights.
manifest_path = next(IN.rglob('v73_reviewed_provenance.json'))
from v73_split_provenance import load_reviewed_manifest, load_fold_mapping, validate_checkpoint_binding
manifest = load_reviewed_manifest(manifest_path, EXPECTED_PROVENANCE_SHA, 2)
if manifest['folds_sha256'] != EXPECTED_FOLDS_SHA:
    raise RuntimeError('reviewed split differs')
test_dirs = [p for p in IN.rglob('test_images') if p.is_dir()]
if len(test_dirs) != 1:
    raise RuntimeError('exactly one actual competition test_images directory required')
test_dir = test_dirs[0]
coco = json.loads(train_json.read_text())
load_fold_mapping(folds_csv, {i['file_name'] for i in coco['images']}, 2, manifest['folds_sha256'])
validate_checkpoint_binding(yolo_ckpt, 'yolo', manifest)
validate_checkpoint_binding(baseline_ckpt, 'baseline', manifest)
# No package installs, CUDA inspection or training until all artifact preflights pass.
subprocess.run([sys.executable, str(code_dir / 'pretrain_unet_encoder_ssl.py'),
                '--corpus', str(corpus_dir), '--train-json', str(train_json),
                '--test-dir', str(test_dir), '--validate-only',
                '--out', '/kaggle/working/unet_encoder_ssl.pt'], check=True)
WORK = Path('/kaggle/working')
for name, p in [('corpus', corpus_dir), ('preproc', preproc_dir),
                ('folds', folds_csv), ('baseline', baseline_ckpt),
                ('yolo', yolo_ckpt), ('train_json', train_json),
                ('code', code_dir)]:
    print(name, '->', p)
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics', 'pycocotools'], check=True)


In [ ]:
%%time
# [1/4] GONG corpus で U-Net encoder を SimCLR 事前学習 (solar3 変換込み)
gpu_env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
r = subprocess.run([sys.executable,
                    str(code_dir / 'pretrain_unet_encoder_ssl.py'),
                    '--corpus', str(corpus_dir),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--epochs', '60', '--batch', '32', '--crop', '320',
                    '--out', str(WORK / 'unet_encoder_ssl.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'unet_encoder_ssl.pt').exists():
    raise RuntimeError(f'SSL pretrain failed rc={r.returncode}')


In [ ]:
%%time
# [2/4] fold2 refiner 学習 (folds 0+1 で学習、fold2 valid) — scratch arm
r = subprocess.run([sys.executable, str(code_dir / 'train_refiner_fold.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--expected-folds-sha256', manifest['folds_sha256'],
                    '--model', 'edgeatt', '--init', 'scratch',
                    '--epochs', '14', '--seed', '3407',
                    '--out', str(WORK / 'edgeatt_scratch_fold2.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'edgeatt_scratch_fold2.pt').exists():
    raise RuntimeError(f'scratch arm failed rc={r.returncode}')


In [ ]:
%%time
# [3/4] 同一レシピ・同一 seed — ssl encoder init arm
if not (WORK / 'unet_encoder_ssl.pt').exists():
    raise RuntimeError('SSL encoder checkpoint missing — pretrain cell failed')
r = subprocess.run([sys.executable, str(code_dir / 'train_refiner_fold.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--expected-folds-sha256', manifest['folds_sha256'],
                    '--model', 'edgeatt',
                    '--init', f'ssl:{WORK}/unet_encoder_ssl.pt',
                    '--epochs', '14', '--seed', '3407',
                    '--out', str(WORK / 'edgeatt_ssl_fold2.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'edgeatt_ssl_fold2.pt').exists():
    raise RuntimeError(f'ssl arm failed rc={r.returncode}')


In [ ]:
%%time
# [4/4] fold2 OOF eval — 3 arm を同一 YOLO 提案・同一後処理で採点
for _p in ('edgeatt_scratch_fold2.pt', 'edgeatt_ssl_fold2.pt'):
    if not (WORK / _p).exists():
        raise RuntimeError(f'refiner checkpoint missing: {_p}')
r = subprocess.run([sys.executable, str(code_dir / 'run_fold_oof_eval.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--provenance-manifest', str(manifest_path),
                    '--provenance-sha256', EXPECTED_PROVENANCE_SHA,
                    '--yolo', str(yolo_ckpt), '--model', 'edgeatt',
                    '--refiner', f'baseline={baseline_ckpt}',
                    '--refiner', f'scratch={WORK}/edgeatt_scratch_fold2.pt',
                    '--refiner', f'ssl={WORK}/edgeatt_ssl_fold2.pt',
                    '--out', str(WORK / 'v73_eval.json')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-4000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'v73_eval.json').exists():
    raise RuntimeError(f'eval failed rc={r.returncode}')


In [ ]:
# 結果サマリ — 事前固定ゲート判定
rep = json.load(open(WORK / 'v73_eval.json'))
arms = rep['arms']
table = {a: r['pq_by_threshold'] for a, r in arms.items()}
print(json.dumps(table, indent=2))
gate = {
  'baseline_pq_0.45': arms['baseline']['pq_by_threshold']['0.45'],
  'scratch_pq_0.45': arms['scratch']['pq_by_threshold']['0.45'],
  'ssl_pq_0.45': arms['ssl']['pq_by_threshold']['0.45'],
  'delta_ssl_minus_scratch': round(
      arms['ssl']['pq_by_threshold']['0.45']
      - arms['scratch']['pq_by_threshold']['0.45'], 6),
  'sanity_baseline_near_0.448':
      abs(arms['baseline']['pq_by_threshold']['0.45'] - 0.4483) < 0.01,
}
print(json.dumps(gate, indent=2))
Path(WORK / 'v73_summary.json').write_text(json.dumps(
    {'eval': rep, 'gate': gate}, indent=2))
